# 모델 1 — 엔진 잔여수명(RUL) 확률 예측

NASA N-CMAPSS 터보팬 데이터로 엔진의 남은 비행 횟수를 확률분포로 예측하고, 신뢰구간과 비용 최적 정비 시점까지 계산한다.

- 실행: GPU 런타임, 셀을 위에서 아래로 실행 (첫 셀에서 Google Drive 마운트 승인)
- 데이터: Drive의 `N-CMAPSS_DS*.h5` 자동 인식
- 산출물: `results_rul_v14_2.json`(평가 지표), `preds_rul_v14_2.npz`(예측 저장 — 마지막 셀의 정책 시뮬레이션에 사용)

In [ ]:
# 설정 및 함수 정의
try:
    from google.colab import drive; drive.mount('/content/drive')
    import glob, os as _os
    _hits = glob.glob('/content/drive/MyDrive/**/N-CMAPSS_DS*.h5', recursive=True)
    DATA_DIR = _os.path.dirname(_hits[0]) if _hits else "/content/drive/MyDrive/제주항공"
except Exception:
    DATA_DIR = "."
SUBSETS = []
CAP = 50; L = 40; DOWN = 8; STRIDE = 20
EPOCHS = 30; BATCH = 256; LR = 1e-3; WD = 1e-2; MAX_WIN = 120000
K_FOLDS = 6; ENSEMBLE_M = 5; WARMUP_MSE = 5; N_REGIMES = 8
USE_REGIME_NORM = True; USE_RESIDUAL = True
USE_HIER = False
ALPHA = 0.10; GROUP_CONDITIONAL = True; MIN_CAL_PER_GROUP = 50
CAL_FRAC = 0.15
LEADTIME = 5
MULTITASK = True; LAMBDA_FAULT = 0.3; LAMBDA_HS = 0.3
LAMBDA_THETA = 0.5
TERM_T = 0.0
FAULT_CLASS_WEIGHT = True

USE_QUANTILE = True
QUANTILES = [0.05, 0.1, 0.5, 0.9, 0.95]
LAMBDA_QUANT = 0.5
USE_HAZARD = True
N_BINS = 26
LAMBDA_HAZ = 0.3
USE_MIXTURE = True
LAMBDA_MIX = 0.3
USE_SEQBAYES = True
USE_CQR = True
C_FAIL = 9.0
C_EARLY = 1.0
BETA_NLL = 0.5
GATE_TEMP = True
SEQ_HS_SWITCH = True
GROUP_BY = 'hs'
PH_FLOOR = 3.0
SAVE_PREDS = True

HYPERS = [(0.05, 1e-2, 1.0e-3), (0.10, 1e-2, 1.0e-3), (0.10, 3e-3, 1.3e-3),
          (0.15, 2e-2, 7.0e-4), (0.20, 1e-2, 1.0e-3)]
MC_DROPOUT_K = 8
RPS_TAIL_W = 2.0; RPS_TAIL_EDGE = 15.0
PINB_W = [2.0, 2.0, 1.0, 1.0, 1.0]
SIG_FLOOR = 1.0
MIN_UNITS_ULQ = 10
PMF_RECAL = True
RECAL_KNOTS = 7
RECAL_PIN = 0.2
SUBSET_OFFSET = 1000; TEST_OFFSET = 50000; SEED0 = 0

import h5py, numpy as np, json, time, os
import torch, torch.nn as nn, torch.nn.functional as Fn
from torch.utils.data import TensorDataset, DataLoader
try:
    from sklearn.cluster import MiniBatchKMeans; HAVE_SK = True
except Exception:
    HAVE_SK = False
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
FCN = {1: '단거리', 2: '중거리', 3: '장거리'}
MODULES = ['Fan', 'LPC', 'HPC', 'HPT', 'LPT']; NMOD = len(MODULES)
HAS_THETA = True
rng = np.random.RandomState(0)

def _dset(d, down, chunk=4_000_000):
    assert chunk % down == 0
    n = d.shape[0]
    return np.concatenate([d[s:s + chunk][::down] for s in range(0, n, chunk)])

def _load(path, split):
    with h5py.File(path, 'r') as f:
        if 'X_s_' + split not in f: return None
        Xs = _dset(f['X_s_' + split], DOWN).astype(np.float32)
        W = _dset(f['W_' + split], DOWN).astype(np.float32)
        Y = _dset(f['Y_' + split], DOWN).astype(np.float32).ravel()
        A = _dset(f['A_' + split], DOWN)
        T = _dset(f['T_' + split], DOWN).astype(np.float32) if 'T_' + split in f else None
    unit = A[:, 0].astype(int); cyc = A[:, 1].astype(int); Fc = A[:, 2].astype(int)
    hs = A[:, 3].astype(int) if A.shape[1] > 3 else np.ones(len(unit), int)
    return Xs, W, Y, unit, cyc, Fc, hs, T

def _localize(path):
    import shutil
    if not path.startswith('/content/drive'): return path
    local = '/content/' + os.path.basename(path)
    src_sz = os.path.getsize(path)
    if os.path.exists(local):
        if os.path.getsize(local) == src_sz: return local
        print(f"  [경고] {os.path.basename(local)} 잘린 사본({os.path.getsize(local)/1e9:.2f}GB != {src_sz/1e9:.2f}GB) → 삭제 후 재복사")
        os.remove(local)
    for att in range(3):
        try:
            print(f"  [복사] {os.path.basename(path)} ({src_sz/1e9:.2f}GB) -> /content (시도 {att+1}) ...")
            shutil.copy(path, local)
            if os.path.getsize(local) != src_sz: raise IOError(f"복사본 크기 불일치 {os.path.getsize(local)} != {src_sz}")
            with h5py.File(local, 'r'):
                pass
            return local
        except Exception as e:
            print(f"    복사 실패: {e}")
            if os.path.exists(local):
                try: os.remove(local)
                except Exception: pass
    raise RuntimeError(f"복사 3회 실패: {path} — 3회 모두 실패면 Drive 원본 손상 의심(크기 {src_sz/1e9:.2f}GB 확인, 재업로드 필요)")

def _theta_dev(T):
    nmod = min(T.shape[1] // 2, NMOD)
    out = np.zeros((len(T), NMOD), np.float32)
    for j in range(nmod): out[:, j] = np.abs(T[:, 2 * j:2 * j + 2] - 1.0).mean(1)
    return out

def _fault_per_unit(T, unit, hs):
    nmod = T.shape[1] // 2; lab = {}
    for u in np.unique(unit):
        m = unit == u; Tu = T[m]; hu = hs[m]; k = max(1, len(Tu) // 10)
        healthy = Tu[hu == 1].mean(0) if (hu == 1).sum() > 0 else Tu[:k].mean(0)
        dev = np.abs(Tu[-k:].mean(0) - healthy); mod = [dev[2 * j:2 * j + 2].sum() for j in range(nmod)]
        lab[u] = int(np.argmax(mod)) if nmod > 0 else 0
    return lab

def load_multi():
    global HAS_THETA
    import glob as _glob
    subs = SUBSETS if SUBSETS else sorted(os.path.basename(p) for p in _glob.glob(os.path.join(DATA_DIR, "N-CMAPSS_DS*.h5")))
    blocks = []; nsens = None; used = []; any_theta = False
    for i, ds in enumerate(subs):
        dpath = f"{DATA_DIR}/{ds}"
        if not os.path.exists(dpath): print(f"  [없음] {ds}"); continue
        try: path = _localize(dpath)
        except Exception as e: print(f"  [건너뜀] {ds}: {e}"); continue
        for s in ('dev', 'test'):
            try: d = _load(path, s)
            except Exception as e: print(f"  [읽기실패] {ds}/{s}: {e}"); continue
            if d is None: continue
            if nsens is None: nsens = d[0].shape[1]
            if d[0].shape[1] != nsens: print(f"  [skip] {ds}/{s}"); continue
            off = i * SUBSET_OFFSET + (TEST_OFFSET if s == 'test' else 0)
            Xs, W, Y, unit, cyc, Fc, hs, T = d
            if T is None: print(f"  [주의] {ds}/{s}: T(θ) 없음 → fault/θ 라벨만 제외(RUL 학습엔 사용)")
            blocks.append((Xs, W, Y, unit + off, cyc, Fc, hs, T)); used.append(f"{ds.split('_')[1].split('.')[0]}/{s}")
            any_theta = any_theta or (T is not None)
    if not blocks: raise FileNotFoundError("N-CMAPSS 서브셋 못 찾음.")
    FLs, THs = [], []
    for (bXs, bW, bY, bU, bC, bF, bH, bT) in blocks:
        if bT is not None:
            flab = _fault_per_unit(bT, bU, bH)
            FLs.append(np.array([flab[u] for u in bU], int)); THs.append(_theta_dev(bT))
        else:
            FLs.append(np.full(len(bU), -1, int)); THs.append(np.full((len(bU), NMOD), np.nan, np.float32))
    cat = lambda j: np.concatenate([b[j] for b in blocks])
    Xs, W, Y, U, C, Fc, H = (cat(j) for j in range(7))
    FL = np.concatenate(FLs); THETA = np.concatenate(THs)
    HAS_THETA = any_theta
    n_units = len(set(U.tolist())); n_valid = len(set(U[FL >= 0].tolist())) if (FL >= 0).any() else 0
    print(f"  [데이터] {used}")
    print(f"  [데이터] 유닛 {n_units} (θ라벨 유닛 {n_valid}) · 데시메이션 1/{DOWN} @로드 · θ건강지표 {'O' if HAS_THETA else 'X'}")
    return (Xs, W, Y, U, C, Fc, H, FL, THETA), NMOD

def _regimes(W, w_mu, w_sd, km):
    return np.zeros(len(W), int) if km is None else km.predict((W - w_mu) / w_sd)

class HierBaseline:
    def fit(self, Xs, reg, unit, healthy, n_reg):
        Cc = Xs.shape[1]; self.h_pop = np.zeros((n_reg, Cc), np.float32)
        self.tau2 = np.full((n_reg, Cc), 1e-6, np.float32); self.s2 = np.ones((n_reg, Cc), np.float32)
        for r in range(n_reg):
            hm = (reg == r) & (healthy == 1); base = Xs[hm] if hm.sum() > 0 else (Xs[reg == r] if (reg == r).sum() > 0 else Xs); self.h_pop[r] = base.mean(0)
            means, wv, ns = [], [], []
            for u in np.unique(unit):
                m = hm & (unit == u)
                if m.sum() >= 2: means.append(Xs[m].mean(0)); wv.append(Xs[m].var(0)); ns.append(int(m.sum()))
            if len(means) >= 2:
                self.s2[r] = np.maximum(np.mean(wv, 0), 1e-6); self.tau2[r] = np.maximum(np.stack(means).var(0) - self.s2[r] / max(np.mean(ns), 1.0), 1e-6)
        return self
    def unit_h(self, Xu, ru, hu, n_reg):
        h = self.h_pop.copy()
        for r in range(n_reg):
            m = (ru == r) & (hu == 1); n = int(m.sum())
            if n >= 1: w = self.tau2[r] / (self.tau2[r] + self.s2[r] / max(n, 1)); h[r] = w * Xu[m].mean(0) + (1 - w) * self.h_pop[r]
        return h

def _windows(feats, Y, unit, cyc, Fc, FL, HS, TH):
    chg = np.where((np.diff(unit) != 0) | (np.diff(cyc) != 0))[0] + 1
    Xw, Yw, Fw, Uw, FLw, HSw, THw, Cw, CEw = [], [], [], [], [], [], [], [], []
    for idx in np.split(np.arange(len(unit)), chg):
        if len(idx) < L: continue
        ds = idx
        for s in range(0, len(ds) - L + 1, STRIDE):
            win = ds[s:s + L]; e = win[-1]
            Xw.append(feats[win]); Yw.append(min(Y[e], CAP)); CEw.append(bool(Y[e] >= CAP))
            Fw.append(int(Fc[e])); Uw.append(int(unit[e]))
            FLw.append(int(FL[e])); HSw.append(int(HS[e])); THw.append(TH[e]); Cw.append(int(cyc[e]))
    return (np.asarray(Xw, np.float32), np.asarray(Yw, np.float32), np.asarray(Fw, int), np.asarray(Uw, int),
            np.asarray(FLw, int), np.asarray(HSw, int), np.asarray(THw, np.float32), np.asarray(Cw, int),
            np.asarray(CEw, bool))

def make_fold_data(pool, train_units, eval_units, cal_units=None):
    Xs, W, Y, U, C, F, H, FL, TH = pool
    trm = np.isin(U, train_units); evm = np.isin(U, eval_units)
    cam = np.isin(U, cal_units) if (cal_units is not None and len(cal_units) > 0) else None
    w_mu = W[trm].mean(0); w_sd = W[trm].std(0) + 1e-6
    km = MiniBatchKMeans(N_REGIMES, random_state=0, n_init=3, batch_size=4096).fit((W[trm] - w_mu) / w_sd) if (HAVE_SK and N_REGIMES > 1) else None
    nreg = N_REGIMES if km is not None else 1; reg_all = _regimes(W, w_mu, w_sd, km)
    rmu = np.zeros((nreg, Xs.shape[1]), np.float32); rsd = np.ones((nreg, Xs.shape[1]), np.float32)
    for r in range(nreg):
        m = (reg_all == r) & trm
        if m.sum() >= 2: rmu[r] = Xs[m].mean(0); rsd[r] = Xs[m].std(0) + 1e-6
    hb = HierBaseline().fit(Xs[trm], reg_all[trm], U[trm], H[trm], nreg) if USE_HIER else None
    tr_smu = Xs[trm].mean(0); tr_ssd = Xs[trm].std(0) + 1e-6

    def feats_for(mask):
        out_idx = np.where(mask)[0]
        z = ((Xs[mask] - rmu[reg_all[mask]]) / rsd[reg_all[mask]]) if USE_REGIME_NORM else (Xs[mask] - tr_smu) / tr_ssd
        parts = [z]; res = np.zeros_like(Xs[mask])
        if USE_RESIDUAL:
            if USE_HIER:
                for u in np.unique(U[mask]):
                    um = U[mask] == u; gi = out_idx[um]; h = hb.unit_h(Xs[gi], reg_all[gi], H[gi], nreg); res[um] = (Xs[gi] - h[reg_all[gi]]) / rsd[reg_all[gi]]
            else:
                for r in range(nreg):
                    rm = reg_all[mask] == r
                    if rm.any(): res[rm] = (Xs[mask][rm] - rmu[r]) / rsd[r]
            parts.append(res)
        parts.append((W[mask] - w_mu) / w_sd)
        return np.concatenate(parts, 1).astype(np.float32)

    def wins(mask):
        return _windows(feats_for(mask), Y[mask], U[mask], C[mask], F[mask], FL[mask], H[mask], TH[mask])

    tr = wins(trm); ev = wins(evm); ca = wins(cam) if cam is not None else None
    if len(tr[0]) > MAX_WIN:
        sel = rng.choice(len(tr[0]), MAX_WIN, replace=False); tr = tuple(a[sel] for a in tr)
    mu = tr[0].reshape(-1, tr[0].shape[-1]).mean(0); sd = tr[0].reshape(-1, tr[0].shape[-1]).std(0) + 1e-6
    nm = lambda A: ((A - mu) / sd).astype(np.float32)
    norm = lambda t: (nm(t[0]),) + t[1:]
    return norm(tr), norm(ev), (norm(ca) if ca is not None else None)

QS = np.array(QUANTILES, np.float32); NQ = len(QUANTILES)
BIN_EDGES = (np.arange(N_BINS) + 1.0) / N_BINS * CAP
Q_LO, Q_HI = 0, NQ - 1

def _bin_idx(y):
    return np.clip((y / CAP * N_BINS).astype(int), 0, N_BINS - 1)

class MultiCNN(nn.Module):
    def __init__(self, C, n_fault, n_theta, p_drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(C, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p_drop),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p_drop),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Dropout(max(p_drop, 0.1) * 2))
        self.mu = nn.Linear(64, 1); self.lv = nn.Linear(64, 1)
        self.quant = nn.Linear(64, NQ)
        self.bins = nn.Linear(64, N_BINS + 1)
        self.fault = nn.Linear(64, max(n_fault, 1)); self.hsh = nn.Linear(64, 1)
        self.theta = nn.Linear(64, n_theta)
        self.mmu = nn.Linear(64, NMOD); self.mlv = nn.Linear(64, NMOD)
    def forward(self, x):
        z = self.head(self.net(x.transpose(1, 2)))
        qr = self.quant(z)
        q = qr[:, :1] + torch.cat([torch.zeros_like(qr[:, :1]), Fn.softplus(qr[:, 1:])], 1).cumsum(1)
        return {'mu': self.mu(z).squeeze(-1), 'lv': self.lv(z).squeeze(-1).clamp(-6, 6),
                'q': q, 'bins': self.bins(z), 'fault': self.fault(z),
                'hs': self.hsh(z).squeeze(-1), 'theta': Fn.softplus(self.theta(z)),
                'mmu': self.mmu(z), 'mlv': self.mlv(z).clamp(-6, 6)}

def term(Ls, t):
    if t <= 0: return Ls.mean()
    return (torch.logsumexp(t * Ls, 0) - np.log(Ls.shape[0])) / t

W_PIN = np.array(PINB_W, np.float32); W_PIN = W_PIN / W_PIN.mean()
def pinball(q, y, qs, w=None):
    e = y.unsqueeze(1) - q
    pb = torch.maximum(qs * e, (qs - 1) * e)
    return (pb * w).mean() if w is not None else pb.mean()

def _loader(arrs, sh=False):
    return DataLoader(TensorDataset(*[torch.tensor(a) for a in arrs]), batch_size=BATCH, shuffle=sh)
def rmse(a, b): return float(np.sqrt(((a - b) ** 2).mean())) if len(a) else float('nan')

def train_one(seed, tr, n_fault, hyper=None):
    Xtr, Ytr, Ftr, Utr, FLtr, HStr, THtr, Ctr, CEtr = tr
    p_drop, wd_h, lr_h = hyper if hyper is not None else (0.1, WD, LR)
    torch.manual_seed(seed); np.random.seed(seed)
    model = MultiCNN(Xtr.shape[-1], n_fault, NMOD, p_drop=p_drop).to(dev)
    fault_w = None
    if MULTITASK and FAULT_CLASS_WEIGHT and (FLtr >= 0).any():
        _cnt = np.bincount(FLtr[FLtr >= 0], minlength=max(n_fault, 1)).astype(np.float32); _cnt[_cnt == 0] = 1.0
        _w = _cnt.sum() / (len(_cnt) * _cnt); fault_w = torch.tensor(_w, dtype=torch.float32).to(dev)
    qs_t = torch.tensor(QS).to(dev)
    Btr = np.where(CEtr, N_BINS, _bin_idx(Ytr)).astype(np.int64)
    opt = torch.optim.AdamW(model.parameters(), lr=lr_h, weight_decay=wd_h); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    w_rps = torch.tensor((1.0 + (RPS_TAIL_W - 1.0) * (BIN_EDGES <= RPS_TAIL_EDGE)).astype(np.float32)).to(dev)
    w_rps = w_rps / w_rps.mean(); w_pin = torch.tensor(W_PIN).to(dev); ar = torch.arange(N_BINS, device=dev)
    for ep in range(EPOCHS):
        model.train(); use_nll = ep >= WARMUP_MSE
        for xb, yb, flb, hsb, thb, bb, cb in _loader((Xtr, Ytr, FLtr, HStr, THtr, Btr, CEtr), sh=True):
            xb, yb, flb, hsb, thb, bb, cb = xb.to(dev), yb.to(dev), flb.to(dev), hsb.float().to(dev), thb.to(dev), bb.to(dev), cb.to(dev)
            opt.zero_grad(); o = model(xb); d = o['mu'] - yb
            if use_nll:
                rul_ps = 0.5 * (o['lv'] + d ** 2 * torch.exp(-o['lv']))
                if BETA_NLL > 0: rul_ps = rul_ps * torch.exp(BETA_NLL * o['lv']).detach()
            else:
                rul_ps = d ** 2
            loss = term(rul_ps, TERM_T)
            if USE_QUANTILE: loss = loss + LAMBDA_QUANT * pinball(o['q'], yb, qs_t, w_pin)
            if USE_HAZARD:
                p = torch.softmax(o['bins'], 1)
                Fc_ = p[:, :N_BINS].cumsum(1)
                tgt = (ar.unsqueeze(0) >= bb.unsqueeze(1)).float()
                loss = loss + LAMBDA_HAZ * ((Fc_ - tgt) ** 2 * w_rps).sum(1).mean()
            if MULTITASK:
                if (flb >= 0).any():
                    loss = loss + LAMBDA_FAULT * Fn.cross_entropy(o['fault'], flb, weight=fault_w, ignore_index=-1)
                loss = loss + LAMBDA_HS * Fn.binary_cross_entropy_with_logits(o['hs'], hsb)
                if HAS_THETA and LAMBDA_THETA > 0:
                    tm = ~torch.isnan(thb).any(1)
                    if tm.any(): loss = loss + LAMBDA_THETA * Fn.mse_loss(o['theta'][tm], thb[tm])
            if USE_MIXTURE:
                vm = flb >= 0
                if vm.any():
                    gi = flb.clamp(min=0)
                    mm = o['mmu'].gather(1, gi.unsqueeze(1)).squeeze(1); ml = o['mlv'].gather(1, gi.unsqueeze(1)).squeeze(1)
                    dm = mm - yb
                    if use_nll:
                        mix_ps = 0.5 * (ml + dm ** 2 * torch.exp(-ml))
                        if BETA_NLL > 0: mix_ps = mix_ps * torch.exp(BETA_NLL * ml).detach()
                    else:
                        mix_ps = dm ** 2
                    loss = loss + LAMBDA_MIX * mix_ps[vm].mean()
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        sch.step()
    return model

def _set_mc(model, on):
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.train(on)

def _predict(model, ev):

    model.eval(); K = max(int(MC_DROPOUT_K), 1)
    out = {k: [] for k in ['mu', 'sig', 'q', 'bins', 'fa', 'hs', 'th', 'mmu', 'msig', 'mcv']}
    with torch.no_grad():
        for xb, *_ in _loader(ev):
            xd = xb.to(dev)
            _set_mc(model, False); o0 = model(xd)
            out['fa'].append(torch.softmax(o0['fault'], 1).cpu().numpy())
            out['hs'].append(torch.sigmoid(o0['hs']).cpu().numpy())
            out['th'].append(o0['theta'].cpu().numpy())
            acc = {}; mus = []
            if K > 1: _set_mc(model, True)
            for _k in range(K):
                o = o0 if K == 1 else model(xd)
                cur = {'mu': o['mu'], 'sig': torch.exp(0.5 * o['lv']), 'q': o['q'],
                       'bins': torch.softmax(o['bins'], 1), 'mmu': o['mmu'],
                       'msig': torch.exp(0.5 * o['mlv'])}
                mus.append(o['mu'])
                for kk, vv in cur.items(): acc[kk] = acc.get(kk, 0) + vv / K
            for kk in acc: out[kk].append(acc[kk].cpu().numpy())
            out['mcv'].append((torch.stack(mus).var(0, unbiased=False) if K > 1 else torch.zeros_like(mus[0])).cpu().numpy())
            _set_mc(model, False)
    return {k: np.concatenate(v) for k, v in out.items()}

def ensemble_predict(models, ev):
    P = [_predict(m, ev) for m in models]
    mu_s = np.stack([p['mu'] for p in P])
    mu = np.clip(mu_s.mean(0), 0, CAP)
    aleat = (np.stack([p['sig'] for p in P]) ** 2).mean(0)
    epist = mu_s.var(0) + np.stack([p['mcv'] for p in P]).mean(0)
    return dict(mu=mu, sig=np.sqrt(aleat + epist), aleat=np.sqrt(aleat), epist=np.sqrt(epist),
                q=np.clip(np.stack([p['q'] for p in P]).mean(0), 0, CAP),
                bins=np.stack([p['bins'] for p in P]).mean(0),
                fa=np.stack([p['fa'] for p in P]).mean(0),
                faM=np.stack([p['fa'] for p in P]),
                hs=np.stack([p['hs'] for p in P]).mean(0),
                th=np.stack([p['th'] for p in P]).mean(0),
                mmu=np.clip(np.stack([p['mmu'] for p in P]).mean(0), 0, CAP),
                msig=np.stack([p['msig'] for p in P]).mean(0))

def cqr_Q(q_lo, q_hi, y, G, alpha, group, keys=(1, 2, 3)):
    def q_of(idx):
        if len(idx) == 0: return 0.0
        E = np.maximum(q_lo[idx] - y[idx], y[idx] - q_hi[idx]); n = len(idx)
        k = min(int(np.ceil((n + 1) * (1 - alpha))), n); return float(np.sort(E)[k - 1])
    if not group: return {c: q_of(np.arange(len(y))) for c in keys}
    return {c: (q_of(np.where(G == c)[0]) if (G == c).sum() >= MIN_CAL_PER_GROUP else q_of(np.arange(len(y)))) for c in keys}

def one_sided_Q(a0, y, beta):
    E = a0 - y; n = len(E)
    if n == 0: return 0.0
    k = min(int(np.ceil((n + 1) * beta)), n); return float(np.sort(E)[k - 1])

def one_sided_Q_group(a0, y, G, beta, keys):
    Qall = one_sided_Q(a0, y, beta)
    return {c: (one_sided_Q(a0[G == c], y[G == c], beta) if (G == c).sum() >= MIN_CAL_PER_GROUP else Qall) for c in keys}

def fit_temp(fa_cal, fl_cal):
    m = fl_cal >= 0
    if m.sum() < 30: return 1.0
    logp = np.log(np.maximum(fa_cal[m], 1e-9)); yv = fl_cal[m]
    best_T, best_nll = 1.0, np.inf
    for T in np.linspace(0.25, 4.0, 16):
        z = logp / T; z = z - z.max(1, keepdims=True)
        p = np.exp(z); p = p / p.sum(1, keepdims=True)
        nll = float(-np.log(np.maximum(p[np.arange(len(yv)), yv], 1e-12)).mean())
        if nll < best_nll: best_nll, best_T = nll, T
    return float(best_T)

def apply_temp(fa, T):
    z = np.log(np.maximum(fa, 1e-9)) / T; z = z - z.max(1, keepdims=True)
    p = np.exp(z); return p / p.sum(1, keepdims=True)

import math
def _Phi(z): return 0.5 * (1 + np.vectorize(math.erf)(z / np.sqrt(2)))
def _phi(z): return np.exp(-z * z / 2) / np.sqrt(2 * np.pi)
def crps_gauss(mu, sig, y):
    sig = np.maximum(sig, 1e-6); z = (y - mu) / sig
    return float((sig * (z * (2 * _Phi(z) - 1) + 2 * _phi(z) - 1 / np.sqrt(np.pi))).mean())
def crps_quant(qmat, y, qs):
    e = y[:, None] - qmat
    return float(2 * np.maximum(qs[None, :] * e, (qs[None, :] - 1) * e).mean())
def winkler(lo, hi, y, alpha):
    s = (hi - lo) + np.where(y < lo, (2 / alpha) * (lo - y), 0.0) + np.where(y > hi, (2 / alpha) * (y - hi), 0.0)
    return float(s.mean())
def picp_mpiw(lo, hi, y): return float(((y >= lo) & (y <= hi)).mean()), float((hi - lo).mean())
def alpha_lambda(mu, y, a=0.2): return float((np.abs(mu - y) <= a * (y + 1e-6)).mean())
def s_score(mu, y):
    d = mu - y; return float(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1).sum())
def ph_units(mu, y, U, a=0.2):
    phs = []
    for u in np.unique(U):
        m = U == u; yu = y[m]; ok = (np.abs(mu[m] - yu) <= np.maximum(a * yu, PH_FLOOR))
        order = np.argsort(yu); oks = ok[order]; ys = yu[order]; t = 0.0
        for i in range(len(ys)):
            if oks[i]: t = ys[i]
            else: break
        phs.append(t)
    return float(np.mean(phs)) if phs else float('nan')
def boot_rmse(mu, y, U, B=300):
    us = np.unique(U); vals = np.array([rmse(mu[U == u], y[U == u]) for u in us])
    r2 = np.random.RandomState(1); rs = np.array([r2.choice(vals, len(vals), replace=True).mean() for _ in range(B)])
    return round(float(rs.mean()), 2), round(float(rs.std()), 2), [round(float(np.percentile(rs, 5)), 2), round(float(np.percentile(rs, 95)), 2)]

BIN_CENTERS = (np.arange(N_BINS) + 0.5) / N_BINS * CAP

def pmf_quantile(bins, tau):
    F = np.cumsum(bins[:, :N_BINS], 1)
    out = np.full(len(bins), float(CAP))
    ks = np.array([np.searchsorted(F[i], tau) for i in range(len(bins))])
    for i, k in enumerate(ks):
        if k >= N_BINS: continue
        F0 = F[i][k - 1] if k > 0 else 0.0; z0 = BIN_EDGES[k - 1] if k > 0 else 0.0
        pk = max(F[i][k] - F0, 1e-12)
        out[i] = z0 + (tau - F0) / pk * (BIN_EDGES[k] - z0)
    return out

def pmf_cdf_at(bins, y):
    F = np.cumsum(bins[:, :N_BINS], 1)
    out = np.empty(len(bins))
    for i in range(len(bins)):
        yi = y[i]
        if yi >= CAP: out[i] = F[i, -1]; continue
        k = np.searchsorted(BIN_EDGES, yi)
        F0 = F[i][k - 1] if k > 0 else 0.0; z0 = BIN_EDGES[k - 1] if k > 0 else 0.0
        out[i] = F0 + (F[i][k] - F0) * (yi - z0) / max(BIN_EDGES[k] - z0, 1e-12)
    return out

def fit_recal(bins_cal, y_cal, cens_cal):
    Fy = pmf_cdf_at(bins_cal, y_cal); n = len(Fy)
    taus = np.linspace(0.0, 1.0, RECAL_KNOTS + 2)
    unc = ~cens_cal
    R = np.zeros_like(taus)
    for j, t in enumerate(taus):
        a = float((Fy[unc] <= t).sum())
        if cens_cal.any():
            Fc = Fy[cens_cal]
            a += float(np.clip((t - Fc) / np.maximum(1.0 - Fc, 1e-9), 0.0, 1.0).sum())
        R[j] = a / max(n, 1)
    R[0] = 0.0; R[-1] = 1.0
    R[taus <= RECAL_PIN] = taus[taus <= RECAL_PIN]
    return taus, np.maximum.accumulate(R)

def apply_recal(bins, taus, R):
    F = np.cumsum(bins[:, :N_BINS], 1)
    F2 = np.interp(F, taus, R)
    p = np.diff(np.concatenate([np.zeros((len(bins), 1)), F2], 1), axis=1)
    out = np.concatenate([p, np.maximum(1.0 - F2[:, -1:], 0.0)], 1)
    out = np.maximum(out, 0.0)
    return out / np.maximum(out.sum(1, keepdims=True), 1e-12)

def renewal_T(bins, cp, cf):
    F = np.cumsum(bins[:, :N_BINS], 1); S = 1.0 - F; h = CAP / N_BINS
    Scum = np.concatenate([np.zeros((len(S), 1)), np.cumsum(S, 1)[:, :-1]], 1)
    D = h * (1.0 + Scum)
    g = (cp * S + cf * (1.0 - S)) / np.maximum(D, 1e-9)
    return BIN_EDGES[g.argmin(1)]

def crps_pmf(bins, y):
    F = np.cumsum(bins[:, :N_BINS], 1)
    tgt = (BIN_EDGES[None, :] >= y[:, None]).astype(np.float64)
    return float(((F - tgt) ** 2).sum(1).mean() * (CAP / N_BINS))

def unit_level_Q(scores, units, beta):
    us = np.unique(units)
    if len(us) >= MIN_UNITS_ULQ:
        su = np.sort([np.quantile(scores[units == u], beta) for u in us])
        k = min(int(np.ceil((len(su) + 1) * beta)), len(su))
        return float(su[k - 1]), f'unit-level(K={len(us)})'
    n = len(scores); k = min(int(np.ceil((n + 1) * beta)), n)
    return float(np.sort(scores)[k - 1]), f'pooled(K={len(us)}<{MIN_UNITS_ULQ})'
def quant_at(qmat, tau): return np.array([np.interp(tau, QS, qmat[i]) for i in range(len(qmat))])
def newsvendor_cost(action, y):
    return float((C_EARLY * np.maximum(y - action, 0) + C_FAIL * np.maximum(action - y, 0)).mean())
def tail_risk_bins(bins, delta): return bins[:, :N_BINS][:, BIN_CENTERS <= delta].sum(1)
def tail_risk_mix(mmu, msig, fa, delta):
    msig = np.maximum(msig, 1e-6); nmod = min(mmu.shape[1], fa.shape[1])
    return (fa[:, :nmod] * _Phi((delta - mmu[:, :nmod]) / msig[:, :nmod])).sum(1)
def seq_bayes(mu, sig, C, U, hs=None):

    out = mu.copy()
    for u in np.unique(U):
        idx = np.where(U == u)[0]; order = idx[np.argsort(C[idx])]
        x = np.array([float(mu[order[0]]), 0.0]); P = np.diag([float(sig[order[0]] ** 2) + 1.0, 4.0])
        Qh = np.diag([1.0, 0.05]); Qd = np.diag([1.0, 0.5]); prev = float(C[order[0]])
        for i in order:
            Qm = Qd if (SEQ_HS_SWITCH and hs is not None and hs[i] < 0.5) else Qh
            dt = max(float(C[i]) - prev, 0.0); prev = float(C[i])
            Fm = np.array([[1.0, dt], [0.0, 1.0]]); x = Fm @ x; P = Fm @ P @ Fm.T + Qm
            r = float(sig[i] ** 2) + 1e-6; H = np.array([1.0, 0.0])
            S = float(H @ P @ H + r); Kk = (P @ H) / S; x = x + Kk * (float(mu[i]) - float(H @ x)); P = P - np.outer(Kk, H @ P)
            out[i] = min(max(x[0], 0.0), CAP)
    return out

def run():
    t0 = time.time(); pool, nmod = load_multi(); U_all = pool[3]
    units = np.array(sorted(set(U_all.tolist()))); rng.shuffle(units); folds = np.array_split(units, min(K_FOLDS, len(units)))
    tau = C_EARLY / (C_EARLY + C_FAIL)
    print(f"[데이터] 유닛 {len(units)} → {len(folds)}-fold | τ*={tau:.2f} | 토글 Q={USE_QUANTILE} HAZ={USE_HAZARD} MIX={USE_MIXTURE} SEQ={USE_SEQBAYES} CQR={USE_CQR}")
    K = ['mu','sig','aleat','epist','q','bins','y','F','U','C','fold','fa','fl','hsp','hst','thp','tht','mmu','msig','lo','hi','act','actq','tstar','cens','faT']
    O = {k: [] for k in K}; temps = []; cal_mode = 'none'
    for k, tu in enumerate(folds):
        tr_all = units[~np.isin(units, tu)]
        r2 = np.random.RandomState(100 + k); perm = r2.permutation(tr_all)
        n_cal = min(max(3, int(round(CAL_FRAC * len(tr_all)))), max(len(tr_all) - 2, 0)) if USE_CQR else 0
        cal_u, fit_u = perm[:n_cal], perm[n_cal:]
        tr, ev, ca = make_fold_data(pool, fit_u, tu, cal_u if n_cal else None)
        models = [train_one(SEED0 + 10 * k + i, tr, nmod, HYPERS[i % len(HYPERS)]) for i in range(ENSEMBLE_M)]
        E = ensemble_predict(models, ev)
        Xe, Ye, Fe, Ue, FLe, HSe, THe, Ce, CEe = ev
        lo, hi = E['q'][:, Q_LO].copy(), E['q'][:, Q_HI].copy()
        sig_e = np.maximum(E['sig'], SIG_FLOOR)
        a0_pmf = pmf_quantile(E['bins'], tau)
        a0_qh = quant_at(E['q'], tau)
        act = a0_pmf.copy(); act_qh = a0_qh.copy(); faT = E['fa']; cal_mode = 'none'
        tstar = renewal_T(E['bins'], C_EARLY, C_FAIL)
        if USE_CQR and ca is not None:
            Eca = ensemble_predict(models, ca); Yca, Fca, Uca = ca[1], ca[2], ca[3]
            sig_c = np.maximum(Eca['sig'], SIG_FLOOR)
            if GROUP_BY == 'hs':
                Ge = (E['hs'] < 0.5).astype(int); Gca = (Eca['hs'] < 0.5).astype(int); gkeys = (0, 1)
            else:
                Ge = Fe; Gca = Fca; gkeys = (1, 2, 3)
            Ec = np.maximum(Eca['q'][:, Q_LO] - Yca, Yca - Eca['q'][:, Q_HI]) / sig_c
            Qc = {c: 0.0 for c in gkeys}
            for c in gkeys:
                gm = Gca == c
                sc_pool = Ec[gm] if int(gm.sum()) >= MIN_CAL_PER_GROUP else Ec
                kq = min(int(np.ceil((len(sc_pool) + 1) * (1 - ALPHA))), len(sc_pool)); Qc[c] = float(np.sort(sc_pool)[kq - 1])
            adj = np.array([Qc[c] for c in Ge]) * sig_e; lo = lo - adj; hi = hi + adj
            s_dec = (pmf_quantile(Eca['bins'], tau) - Yca) / sig_c
            Qd, cal_mode = unit_level_Q(s_dec, Uca, 1 - tau)
            act = a0_pmf - Qd * sig_e
            Qd2, _ = unit_level_Q((quant_at(Eca['q'], tau) - Yca) / sig_c, Uca, 1 - tau)
            act_qh = a0_qh - Qd2 * sig_e
            FLca = ca[4]; vca = FLca >= 0

            if vca.any():
                uca = sorted(set(ca[3][vca].tolist())); accs = []
                for mi in range(Eca['faM'].shape[0]):
                    pm = Eca['faM'][mi].argmax(1)
                    ok = [int(np.bincount(pm[ca[3] == u]).argmax()) == int(FLca[ca[3] == u][0]) for u in uca]
                    aw = float((pm[vca] == FLca[vca]).mean())
                    accs.append(float(np.mean(ok)) + 0.001 * aw)
                best = int(np.argmax(accs)); fa_sel = E['faM'][best]
                print(f"    [fault 멤버선택] cal 정확도 {['%.3f' % a for a in accs]} → 멤버 {best}")
            else:
                fa_sel = E['fa']; accs = []; best = -1
            E['fa'] = fa_sel
            if GATE_TEMP:
                Tg = fit_temp(Eca['faM'][best] if best >= 0 else Eca['fa'], ca[4]); faT = apply_temp(fa_sel, Tg); temps.append(Tg)
            if PMF_RECAL:
                tks, Rv = fit_recal(Eca['bins'], Yca, ca[8])
                E['bins'] = apply_recal(E['bins'], tks, Rv)
                tstar = renewal_T(E['bins'], C_EARLY, C_FAIL)
        lo = np.clip(lo, 0, CAP); hi = np.clip(hi, 0, CAP)
        act = np.clip(act, 0, CAP); act_qh = np.clip(act_qh, 0, CAP)
        for key, val in [('mu',E['mu']),('sig',E['sig']),('aleat',E['aleat']),('epist',E['epist']),('q',E['q']),
                         ('bins',E['bins']),('y',Ye),('F',Fe),('U',Ue),('C',Ce),('fold',np.full(len(Ye),k)),
                         ('fa',E['fa']),('fl',FLe),('hsp',(E['hs']>0.5).astype(int)),('hst',HSe),
                         ('thp',E['th']),('tht',THe),('mmu',E['mmu']),('msig',E['msig']),
                         ('lo',lo),('hi',hi),('act',act),('actq',act_qh),('tstar',tstar),('cens',CEe),('faT',faT)]:
            O[key].append(val)
        print(f"  fold {k+1}/{len(folds)}: 학습 {len(fit_u)}·보정 {n_cal}·평가 {len(tu)} 유닛 · win {len(Ye)} [{time.time()-t0:.0f}s]")
    for k in O: O[k] = np.concatenate(O[k])
    mu, sig, y, F, U, C, fold = O['mu'], O['sig'], O['y'], O['F'], O['U'], O['C'], O['fold']
    m_deg = ~O['cens']; m_eol = y <= 15

    uinfo = {u: (int(F[U == u][0]), rmse(mu[U == u], y[U == u])) for u in sorted(set(U.tolist()))}
    allr = [r for _, r in uinfo.values()]
    overall = [round(rmse(mu, y), 2), round(float(np.mean(allr)), 2), round(float(np.std(allr)), 2)]
    rmse_deg = round(rmse(mu[m_deg], y[m_deg]), 2); rmse_eol = round(rmse(mu[m_eol], y[m_eol]), 2) if m_eol.any() else None
    bmean, bstd, bci = boot_rmse(mu, y, U)
    per = {}
    for c in (1, 2, 3):
        v = [r for (cl, r) in uinfo.values() if cl == c]
        per[FCN[c]] = [round(float(np.mean(v)), 2), round(float(np.std(v)), 2), len(v)] if v else [float('nan'), 0, 0]

    picp_raw, mpiw_raw = picp_mpiw(np.clip(O['q'][:, Q_LO], 0, CAP), np.clip(O['q'][:, Q_HI], 0, CAP), y)
    picp_c, mpiw_c = picp_mpiw(O['lo'], O['hi'], y)
    wink = round(winkler(O['lo'], O['hi'], y, ALPHA), 2)
    picp_deg, mpiw_deg = picp_mpiw(O['lo'][m_deg], O['hi'][m_deg], y[m_deg])
    wink_deg = round(winkler(O['lo'][m_deg], O['hi'][m_deg], y[m_deg], ALPHA), 2)
    picp_eol = round(picp_mpiw(O['lo'][m_eol], O['hi'][m_eol], y[m_eol])[0], 3) if m_eol.any() else None

    crps_q = round(crps_quant(O['q'], y, QS), 3); crps_q_deg = round(crps_quant(O['q'][m_deg], y[m_deg], QS), 3)
    crps_g = round(crps_gauss(mu, sig, y), 3)
    crps_p = round(crps_pmf(O['bins'], y), 3); crps_p_deg = round(crps_pmf(O['bins'][m_deg], y[m_deg]), 3)
    calib = {}
    for j, t in enumerate(QUANTILES):
        qp = pmf_quantile(O['bins'], t)
        calib[str(t)] = {'pmf': round(float((y <= qp).mean()), 3),
                         'pmf_deg': round(float((y[m_deg] <= qp[m_deg]).mean()), 3),
                         'qhead_deg': round(float((y[m_deg] <= O['q'][m_deg][:, j]).mean()), 3)}
    al = round(alpha_lambda(mu, y), 3); ph = round(ph_units(mu, y, U), 2); ss = round(s_score(mu, y), 1)
    ssm = round(ss / max(len(y), 1), 3)
    aleat_m = round(float(O['aleat'].mean()), 2); epist_m = round(float(O['epist'].mean()), 2)

    cost_mean = round(newsvendor_cost(mu, y), 3)
    cost_pmf = round(newsvendor_cost(O['act'], y), 3)
    cost_qh = round(newsvendor_cost(O['actq'], y), 3)
    cost_ts = round(newsvendor_cost(np.clip(O['tstar'], 0, CAP), y), 3)
    cost_red = round((cost_mean - cost_pmf) / max(cost_mean, 1e-9) * 100, 1)
    cost_mean_deg = round(newsvendor_cost(mu[m_deg], y[m_deg]), 3); cost_qc_deg = round(newsvendor_cost(O['act'][m_deg], y[m_deg]), 3)
    viol = round(float((O['act'] > y).mean()), 3)
    viol_deg = round(float((O['act'][m_deg] > y[m_deg]).mean()), 3)
    viol_qh_deg = round(float((O['actq'][m_deg] > y[m_deg]).mean()), 3)
    beta_sd = round(float(np.sqrt(tau * (1 - tau) / (MIN_UNITS_ULQ + 2))), 3)
    tail = tail_risk_bins(O['bins'], LEADTIME).mean() if USE_HAZARD else _Phi((LEADTIME - mu) / np.maximum(sig, 1e-6)).mean()
    tail = round(float(tail), 3)

    seq_rmse = None; seq_rmse_deg = None
    if USE_SEQBAYES:
        mu_s = seq_bayes(mu, sig, C, U, O['hsp']); seq_rmse = round(rmse(mu_s, y), 2); seq_rmse_deg = round(rmse(mu_s[m_deg], y[m_deg]), 2)

    mix_rmse = None; mix_tail = None
    if USE_MIXTURE:
        w = O['faT'][:, :nmod] / (O['faT'][:, :nmod].sum(1, keepdims=True) + 1e-9)
        mix_mu = np.clip((w * O['mmu'][:, :nmod]).sum(1), 0, CAP)
        mix_rmse = round(rmse(mix_mu, y), 2)
        mix_tail = round(float(tail_risk_mix(O['mmu'], O['msig'], O['faT'], LEADTIME).mean()), 3)

    from collections import Counter as _C
    fa_pred, fl_true = O['fa'].argmax(1), O['fl']
    uall = sorted(set(U.tolist()))
    utrue = {u: int(fl_true[U == u][0]) for u in uall if int(fl_true[U == u][0]) >= 0}
    upred = {u: int(np.bincount(fa_pred[U == u]).argmax()) for u in utrue}
    fault_acc = round(sum(upred[u] == utrue[u] for u in utrue) / max(len(utrue), 1), 3) if utrue else None
    cnt = _C(utrue.values()); fault_dist = {MODULES[c]: cnt[c] for c in range(NMOD) if cnt.get(c, 0) > 0}
    fault_base = round(max(fault_dist.values()) / sum(fault_dist.values()), 3) if fault_dist else None
    hs_acc = round(float((O['hsp'] == O['hst']).mean()), 3)
    tvm = ~np.isnan(O['tht']).any(1)
    theta_rmse = round(float(np.sqrt(((O['thp'][tvm] - O['tht'][tvm]) ** 2).mean())), 4) if tvm.any() else None

    print(f"\n===== RUL v14.2 (유닛교차 {len(folds)}fold · M{ENSEMBLE_M} · split-CQR) =====")
    print(f"  [RUL] pool {overall[0]} / 유닛 {overall[1]}±{overall[2]} · 부트스트랩 {bmean}±{bstd} 90%CI{bci}")
    print(f"        층화: 열화구간(y<CAP) {rmse_deg} · 근접EOL(y≤15) {rmse_eol} · 클래스별 {per}")
    print(f"        ※ 유닛교차 프로토콜 — 공식 dev/test split 발표수치와 직접 비교 불가")
    print(f"  [구간] 보정전 {round(picp_raw,3)}(폭 {round(mpiw_raw,1)}) → split-CQR후 {round(picp_c,3)}(폭 {round(mpiw_c,1)}) · Winkler {wink}")
    print(f"        층화: 열화구간 커버리지 {round(picp_deg,3)}(폭 {round(mpiw_deg,1)}, Winkler {wink_deg}) · 근접EOL {picp_eol}")
    print(f"        ※ 같은-모델 보정유닛 사용(근사 보정; 윈도우 종속성 탓 이론 보장은 아님)")
    print(f"  [확률지표] CRPS(PMF, 주) {crps_p} (열화 {crps_p_deg}) · CRPS(분위수) {crps_q} (열화 {crps_q_deg}) · 가우시안(참고) {crps_g}")
    print(f"  [보정곡선] " + " · ".join(f"τ{t}: PMF열화 {v['pmf_deg']}/헤드열화 {v['qhead_deg']}" for t, v in calib.items()))
    print(f"        α-λ정확도 {al} · PrognosticHorizon {ph}(바닥 {PH_FLOOR}) · S-score 평균 {ssm} (합계 {ss})")
    print(f"  [불확실성분해] 우연적 {aleat_m} · 인식적 {epist_m}")
    print(f"  [결정] τ*={round(tau,2)} · 보정방식 {cal_mode} (studentized, σ바닥 {SIG_FLOOR})")
    print(f"        비용: 평균 {cost_mean} → PMF분위수(정본) {cost_pmf} ({cost_red}%↓) · 분위수헤드 {cost_qh} · 갱신보상T* {cost_ts}")
    print(f"        열화구간 {cost_mean_deg}→{cost_qc_deg} · 평균 T* = {round(float(np.mean(O['tstar'])),1)}사이클")
    print(f"        경험적 P(정비 전 고장) = {viol} (열화 {viol_deg}, 헤드 {viol_qh_deg}, 목표 ≤{round(tau,2)}) · 꼬리위험 P(RUL≤{LEADTIME})={tail}")
    print(f"        ※ 유닛수 유한표본 노이즈: K=10 기준 위반율 sd≈{beta_sd} — 정확 도달이 아니라 근접이 목표")
    if USE_SEQBAYES: print(f"  [순차베이즈] RMSE {overall[0]} → {seq_rmse} (열화구간 {rmse_deg} → {seq_rmse_deg})")
    gate_T = round(float(np.mean(temps)), 2) if temps else None
    if USE_MIXTURE: print(f"  [혼합RUL] 단일 {overall[0]} vs 혼합 {mix_rmse} · 혼합 꼬리위험 {mix_tail} · 게이트온도 {gate_T}")
    print(f"  [고장] baseline {fault_base} · 정확도 {fault_acc} · 분포 {fault_dist} | [열화] hs {hs_acc} | [θ] RMSE {theta_rmse}")

    if SAVE_PREDS:
        np.savez_compressed('preds_rul_v14_2.npz', **O)
        print("예측 저장: preds_rul_v14_2.npz (이후 보정·지표 변경은 재학습 불필요)")
    json.dump({"model": "v14.2-final", "n_units": len(units),
               "rmse_pool": overall[0], "rmse_unit_mean": overall[1], "rmse_unit_std": overall[2],
               "rmse_degraded": rmse_deg, "rmse_eol": rmse_eol,
               "rmse_bootstrap_mean": bmean, "rmse_bootstrap_std": bstd, "rmse_bootstrap_90ci": bci,
               "rmse_by_class": per, "coverage_raw": round(picp_raw, 3), "coverage_cqr": round(picp_c, 3),
               "coverage_degraded": round(picp_deg, 3), "coverage_eol": picp_eol,
               "mpiw_cqr": round(mpiw_c, 2), "mpiw_degraded": round(mpiw_deg, 2),
               "winkler": wink, "winkler_degraded": wink_deg,
               "crps_pmf": crps_p, "crps_pmf_degraded": crps_p_deg, "calibration_curve": calib,
               "crps_quantile": crps_q, "crps_quantile_degraded": crps_q_deg, "crps_gauss_ref": crps_g,
               "alpha_lambda": al, "prognostic_horizon": ph, "ph_floor": PH_FLOOR, "s_score": ss, "s_score_mean": ssm, "group_by": GROUP_BY,
               "aleatoric": aleat_m, "epistemic": epist_m,
               "newsvendor_tau": round(tau, 3), "cost_mean_action": cost_mean, "cost_pmf_action": cost_pmf,
               "cost_qhead_action": cost_qh, "cost_tstar_action": cost_ts, "cost_reduction_pct": cost_red,
               "tstar_mean": round(float(np.mean(O['tstar'])), 1), "calibration_mode": cal_mode,
               "cost_mean_degraded": cost_mean_deg, "cost_conformal_degraded": cost_qc_deg,
               "action_violation_rate": viol, "action_violation_degraded": viol_deg, "action_violation_qhead_deg": viol_qh_deg,
               "ensemble_m": ENSEMBLE_M, "mc_dropout_k": MC_DROPOUT_K, "pmf_recal": PMF_RECAL, "recal_pin": RECAL_PIN, "fault_member_select": True,
               "det_class_heads": True,
               "tail_risk": tail, "seqbayes_rmse": seq_rmse, "seqbayes_rmse_degraded": seq_rmse_deg,
               "mixture_rmse": mix_rmse, "mixture_tail_risk": mix_tail, "fault_baseline": fault_base,
               "fault_accuracy": fault_acc, "fault_distribution": fault_dist, "hs_accuracy": hs_acc,
               "theta_rmse": theta_rmse, "gate_temp_mean": gate_T, "beta_nll": BETA_NLL,
               "notes": ["유닛교차 프로토콜: 공식 split 발표수치와 직접 비교 불가",
                          "커버리지·위반율은 studentized 컨포멀 근사 보정 — 이론 보장 아님(윈도우 종속)",
                          "유닛별 조건부 커버리지는 분포무관 보장 불가(Barber et al. 2021) — 경험 보고",
                          "y=CAP 윈도우는 우측절단 처리 · RPS는 절단=오버플로 빈 특수경우라 proper",
                          "T*는 비용률(갱신-보상) 최적, 뉴스벤더 분위수는 단발 위험 제약 — 목적함수 다름"]},
              open("results_rul_v14_2.json", "w"), ensure_ascii=False, indent=2)
    print("\n저장: results_rul_v14_2.json"); return overall, fault_acc, crps_q


In [ ]:
# 데이터 확인
pool, nmod = load_multi()
Xs, W, Y, U, C, Fc, H, FL, TH = pool
from collections import Counter
uf = {u: int(FL[U == u][0]) for u in set(U.tolist())}
cnt = Counter(uf.values())
print('유닛:', len(uf), '· 센서:', Xs.shape[1], '· θ라벨 없는 유닛:', cnt.get(-1, 0))
print('고장 분포:', {MODULES[c]: n for c, n in cnt.items() if c >= 0})
print('토글  Q=%s HAZ=%s MIX=%s SEQ=%s CQR=%s' % (USE_QUANTILE, USE_HAZARD, USE_MIXTURE, USE_SEQBAYES, USE_CQR))
print('비용 C_FAIL:C_EARLY = %g:%g → 뉴스벤더 τ* = %.2f (낮은 분위수 = 보수적 행동)' % (C_FAIL, C_EARLY, C_EARLY / (C_EARLY + C_FAIL)))
print('→ 이제 3번 학습 셀로.')


In [ ]:
# 학습 및 평가
import shutil
overall, fault_acc, crps_q = run()
for fn in ('results_rul_v14_2.json', 'preds_rul_v14_2.npz'):
    try:
        shutil.copy(fn, DATA_DIR); print('-> Drive:', DATA_DIR + '/' + fn)
    except Exception as e:
        print('Drive 복사 건너뜀(%s):' % fn, e)


In [ ]:
# 정책 시뮬레이션 (예측 파일 기반)
import numpy as np, glob, os
cand = ['preds_rul_v14_2.npz'] + glob.glob(os.path.join(DATA_DIR, 'preds_rul_v14_2.npz'))
z = np.load([p for p in cand if os.path.exists(p)][0])
y, U, C, act, mu, cens, tstar = z['y'], z['U'], z['C'], z['act'], z['mu'], z['cens'], z['tstar']
deg = ~cens

print('=== 위반율 해부 (열화구간, y 버킷별) ===')
for lo_, hi_ in [(0, 5), (5, 10), (10, 20), (20, 35), (35, 50)]:
    m = deg & (y >= lo_) & (y < hi_)
    if m.any(): print('  y∈[%2d,%2d): %.3f (n=%d)%s' % (lo_, hi_, (act[m] > y[m]).mean(), m.sum(),
                      '  ← y<5에선 정의상 흔함(운영 지표 아님)' if hi_ == 5 else ''))

print('\n=== 정책 시뮬레이션 (매 사이클 재예측, 트리거 = 행동값 ≤ %d) + Kamariotis M ===' % LEADTIME)
units = sorted(set(U.tolist()))
def simulate(score, thr=float(LEADTIME)):
    fails, waste, cost, cycles = [], [], 0.0, 0.0
    for u in units:
        m = U == u; o = np.argsort(C[m]); yy, ss, cc = y[m][o], score[m][o], C[m][o]
        life = float(cc[-1] + yy[-1])
        t = np.where(ss <= thr)[0]
        if len(t) == 0 or yy[t[0]] <= 0:
            fails.append(u); cost += C_FAIL; cycles += life
        else:
            waste.append(float(yy[t[0]])); cost += C_EARLY; cycles += float(cc[t[0]])
    R = cost / max(cycles, 1e-9)
    R_perf = C_EARLY * len(units) / sum(float(C[U == u].max() + y[U == u][np.argmax(C[U == u])]) for u in units)
    M = (R - R_perf) / max(R_perf, 1e-12)
    return fails, waste, R, M
for name, s in [('PMF분위수(τ*=0.1, 정본)', act), ('갱신보상 T*', tstar), ('평균 μ(순진)', mu)]:
    f, w, R, M = simulate(s)
    print('  %s: 고장 %d/%d %s · 트리거시 잔여 평균 %.1f/min %.0f · 비용률 R %.4f · Kamariotis M %.2f' %
          (name, len(f), len(units), f if len(f) <= 4 else '', np.mean(w) if w else float('nan'),
           min(w) if w else float('nan'), R, M))
print('  (M = 완전정보 대비 비용률 격차 — 작을수록 좋음. T*와 분위수는 목적함수가 다름: 비용률 vs 단발위험)')

print('\n=== 임계값 스윕 (운영점 선택) ===')
for thr in (3, 5, 8, 10):
    row = []
    for name, s in [('PMF', act), ('T*', tstar), ('μ', mu)]:
        f, w, R, M = simulate(s, float(thr))
        row.append('%s %d고장/낭비%.1f/M%.2f' % (name, len(f), np.mean(w) if w else float('nan'), M))
    print('  thr=%2d | %s' % (thr, ' | '.join(row)))

print('\n=== 모델 맹점 유닛 (열화 평균오차 상위 5) ===')
rows = [(float((mu[(U == u) & deg] - y[(U == u) & deg]).mean()), u) for u in units if ((U == u) & deg).sum() > 50]
for b, u in sorted(rows)[-5:]: print('  유닛 %d: %+.1f사이클 (양수=위험 방향)' % (u, b))
print('\n해석: 유닛별 편향은 한계적 보정으론 불가(Barber 2021) — fleet 확대가 해법.')

print('\n=== 결정 이원화: Tier B 유닛수준 안전 보정 (교차폴드 후처리 — 재학습 불필요) ===')

sig_, bins_, fold_ = z['sig'], z['bins'], z['fold']
NBc = N_BINS; EDGc = (np.arange(NBc) + 1.0) / NBc * CAP
def _pq(b, tau):
    F = np.cumsum(b[:, :NBc], 1); out = np.full(len(b), float(CAP))
    for i in range(len(b)):
        k = np.searchsorted(F[i], tau)
        if k >= NBc: continue
        F0 = F[i][k-1] if k else 0.0; z0 = EDGc[k-1] if k else 0.0
        out[i] = z0 + (tau - F0) / max(F[i][k] - F0, 1e-12) * (EDGc[k] - z0)
    return out
a0_ = _pq(bins_, 0.1); sg_ = np.maximum(sig_, 1.0); s_ = (a0_ - y) / sg_
actU = np.empty(len(y))
for k in sorted(set(fold_.tolist())):
    te = fold_ == k; cm = ~te
    us_ = sorted(set(U[cm].tolist()))
    su = np.sort([np.quantile(s_[cm & (U == u)], 0.9) for u in us_])
    kq = min(int(np.ceil((len(su) + 1) * 0.9)), len(su))
    actU[te] = a0_[te] - float(su[kq - 1]) * sg_[te]
actU = np.clip(actU, 0, CAP)
cost_ = lambda a: float((C_EARLY * np.maximum(y - a, 0) + C_FAIL * np.maximum(a - y, 0)).mean())
for name, a in [('Tier A(한계적, 폴드내 보정)', act), ('Tier B(유닛수준 안전, 교차폴드)', actU)]:
    fv = [(a[(fold_ == k) & deg] > y[(fold_ == k) & deg]).mean() for k in sorted(set(fold_.tolist()))]
    print('  %s: 열화 위반 %.3f · 폴드범위 [%.2f, %.2f] · 단발비용 %.2f' %
          (name, (a[deg] > y[deg]).mean(), min(fv), max(fv), cost_(a)))
for thr in (5, 8):
    f, w, R_, M = simulate(actU, float(thr))
    print('  Tier B 정책 thr=%d: 고장 %d/%d %s · 낭비 %.1f · M %.2f' % (thr, len(f), len(units), f if len(f) <= 4 else '', np.mean(w) if w else float('nan'), M))
print('  ※ Tier B는 교차폴드(모델 상이) 근사 + 유닛수준 보정 — fleet 평균이 아닌 엔진별 위험 관리.')
print('    비싸지만(조기교체↑) 맹점 유닛까지 포섭 — 항공사가 고를 두 운영점.')
